In [0]:
# Transform raw support tickets into customer-level support features.

# Import Required Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Bronze Support Table 
support = spark.table("support_tickets_200_k")

In [0]:
# Inspect the Data , Display records 
display(support)


TicketID,CustomerID,TicketDate,Category,Status
TK0,103995,2025-01-07,Technical,Closed
TK1,134936,2025-08-20,Billing,Closed
TK2,132070,2024-11-28,Login,Open
TK3,161308,2024-02-29,Technical,Closed
TK4,157275,2025-10-17,Billing,Closed
TK5,165885,2025-01-19,Account,Closed
TK6,199202,2024-09-01,Technical,Open
TK7,184558,2024-08-07,Technical,Closed
TK8,145583,2025-07-14,Technical,Closed
TK9,197379,2025-07-21,Login,Closed


In [0]:
support.printSchema()

root
 |-- TicketID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TicketDate: date (nullable = true)
 |-- Category: string (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
# Count Total Records 
print("Total Support Records:", support.count())

Total Support Records: 200000


In [0]:
# Check Missing Values 
support.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in support.columns
]).show()

+--------+----------+----------+--------+------+
|TicketID|CustomerID|TicketDate|Category|Status|
+--------+----------+----------+--------+------+
|       0|         0|         0|       0|     0|
+--------+----------+----------+--------+------+



In [0]:
# Remove Duplicate Tickets

# Each support ticket should appear only once. 
silver_support = support.dropDuplicates(["TicketID"])

In [0]:
print("Original Records:", support.count())

print("After Removing Duplicates:", silver_support.count())

Original Records: 200000
After Removing Duplicates: 200000


In [0]:
# Handle Missing Values

# Replace missing category and status. 
silver_support = silver_support.fillna({
    "Category": "UNKNOWN",
    "Status": "OPEN"
})

In [0]:
# Standardize Text Columns

# Remove extra spaces 

silver_support = silver_support.withColumn(
    "Category",
    trim(col("Category"))
)

silver_support = silver_support.withColumn(
    "Status",
    trim(col("Status"))
)

In [0]:
# Convert to uppercase 
silver_support = silver_support.withColumn(
    "Category",
    upper(col("Category"))
)

silver_support = silver_support.withColumn(
    "Status",
    upper(col("Status"))
)

In [0]:
# Convert Ticket Date, Convert string to date. 
silver_support = silver_support.withColumn(
    "TicketDate",
    to_date(col("TicketDate"))
)

In [0]:
# Verify 
silver_support.printSchema()

root
 |-- TicketID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TicketDate: date (nullable = true)
 |-- Category: string (nullable = false)
 |-- Status: string (nullable = false)



In [0]:
# Remove Invalid Dates (Remove records where the date conversion failed) 

silver_support = silver_support.filter(
    col("TicketDate").isNotNull()
)

In [0]:
# Create Customer-Level Metrics (Aggregate ticket information by customer) 
silver_support = silver_support.groupBy("CustomerID").agg(

    count("TicketID").alias("TotalTickets"),

    max("TicketDate").alias("LastTicketDate"),

    sum(
        when(col("Status") == "OPEN", 1).otherwise(0)
    ).alias("OpenTickets"),

    sum(
        when(col("Status") == "CLOSED", 1).otherwise(0)
    ).alias("ClosedTickets")
)

In [0]:
# Create Complaint Score

# Customers with more tickets generally have more complaints. 
silver_support = silver_support.withColumn(

    "ComplaintScore",

    col("TotalTickets")*2 +

    col("OpenTickets")*5
)

In [0]:
# Create Support Category 
silver_support = silver_support.withColumn(

    "SupportLevel",

    when(col("TotalTickets") >= 10, "HIGH")

    .when(col("TotalTickets") >= 5, "MEDIUM")

    .otherwise("LOW")
)

In [0]:
# Open Ticket Ratio

# Measures unresolved issues. 
silver_support = silver_support.withColumn(

    "OpenTicketRatio",

    when(

        col("TotalTickets") == 0,

        0

    ).otherwise(

        round(

            col("OpenTickets") /

            col("TotalTickets"),

            2

        )

    )
)

In [0]:
# Customer Satisfaction Indicator 
silver_support = silver_support.withColumn(

    "SupportRisk",

    when(col("ComplaintScore") >= 30, "HIGH")

    .when(col("ComplaintScore") >= 15, "MEDIUM")

    .otherwise("LOW")
)

In [0]:
# Verify Output 
display(silver_support)

CustomerID,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk
103995,2,2025-01-07,0,2,4,LOW,0.0,LOW
134936,2,2025-08-20,0,2,4,LOW,0.0,LOW
132070,4,2025-11-08,1,3,13,LOW,0.25,LOW
161308,2,2024-10-06,0,2,4,LOW,0.0,LOW
157275,2,2025-10-17,0,2,4,LOW,0.0,LOW
165885,2,2025-11-25,0,2,4,LOW,0.0,LOW
199202,4,2025-06-04,2,2,18,LOW,0.5,MEDIUM
184558,1,2024-08-07,0,1,2,LOW,0.0,LOW
145583,2,2025-07-14,0,2,4,LOW,0.0,LOW
197379,5,2025-10-05,0,5,10,MEDIUM,0.0,LOW


In [0]:
# Verify Schema 
silver_support.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- TotalTickets: long (nullable = false)
 |-- LastTicketDate: date (nullable = true)
 |-- OpenTickets: long (nullable = true)
 |-- ClosedTickets: long (nullable = true)
 |-- ComplaintScore: long (nullable = true)
 |-- SupportLevel: string (nullable = false)
 |-- OpenTicketRatio: double (nullable = true)
 |-- SupportRisk: string (nullable = false)



In [0]:
# Save Silver Support 
silver_support.write.mode("overwrite").saveAsTable("silver_support")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6369503641629216>, line 2
      1 # Save Silver Support 
----> 2 silver_support.write.mode("overwrite").saveAsTable("silver_support")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_comma

In [0]:
spark.table("silver_support").printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- TicketCount: long (nullable = true)
 |-- ResolvedTickets: long (nullable = true)
 |-- ResolutionRate: double (nullable = true)
 |-- ComplaintScore: long (nullable = true)



In [0]:
silver_support.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- TotalTickets: long (nullable = false)
 |-- LastTicketDate: date (nullable = true)
 |-- OpenTickets: long (nullable = true)
 |-- ClosedTickets: long (nullable = true)
 |-- ComplaintScore: long (nullable = true)
 |-- SupportLevel: string (nullable = false)
 |-- OpenTicketRatio: double (nullable = true)
 |-- SupportRisk: string (nullable = false)



In [0]:
spark.sql("DROP TABLE IF EXISTS silver_support")

DataFrame[]

In [0]:
spark.sql("SHOW TABLES").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|billing_transacti...|      false|
| default|      bronze_billing|      false|
| default|        bronze_churn|      false|
| default|    bronze_customers|      false|
| default|      bronze_support|      false|
| default|        bronze_usage|      false|
| default|  churn_labels_100_k|      false|
| default| crm_customers_100_k|      false|
| default|customer_usage_500_k|      false|
| default|      silver_billing|      false|
| default|        silver_churn|      false|
| default|    silver_customers|      false|
| default|        silver_usage|      false|
| default|support_tickets_2...|      false|
+--------+--------------------+-----------+



In [0]:
silver_support.write \
    .mode("overwrite") \
    .saveAsTable("silver_support")

In [0]:
# Validate Saved Table 
display(spark.table("silver_support"))

CustomerID,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk
114159,4,2025-12-23,1,3,13,LOW,0.25,LOW
141799,3,2025-11-28,1,2,11,LOW,0.33,LOW
188310,3,2025-11-18,1,2,11,LOW,0.33,LOW
166790,2,2025-07-30,0,2,4,LOW,0.0,LOW
122750,2,2025-04-23,0,2,4,LOW,0.0,LOW
112810,3,2025-04-24,2,1,16,LOW,0.67,MEDIUM
167559,3,2025-04-16,1,2,11,LOW,0.33,LOW
188596,1,2025-04-14,0,1,2,LOW,0.0,LOW
144095,5,2025-10-26,2,3,20,MEDIUM,0.4,MEDIUM
172162,4,2025-09-22,1,3,13,LOW,0.25,LOW
